In [ ]:
pip install moviepy

In [ ]:
import librosa
import numpy as np
from scipy.signal import find_peaks
from moviepy.editor import VideoFileClip
import os

def get_audio_spikes(video_path, distance_sec=10, threshold_multiplier=3.0):
    """
    Extracts audio from a video file and identifies timestamps with loud energy spikes.

    Args:
        video_path (str): Path to the video file.
        distance_sec (int): Minimum distance between spikes in seconds.
        threshold_multiplier (float): Multiplier for the standard deviation to set the threshold.

    Returns:
        list: A list of timestamps (in seconds) indicating when audio spikes occurred.
    """
    print("Audio Module: Extracting audio stream from video...")
    video = VideoFileClip(video_path)

    # We extract the audio to a temporary wav file that librosa CAN read
    temp_audio = "temp_audio_module.wav"
    video.audio.write_audiofile(temp_audio, codec='pcm_s16le', logger=None)

    # Now load that temp file with librosa
    y, sr = librosa.load(temp_audio, sr=22050)
    os.remove(temp_audio) # Clean up the temp file

    print("Audio Module: Analyzing audio energy for sudden spikes...")
    rms = librosa.feature.rms(y=y)[0]
    rms_smoothed = np.convolve(rms, np.ones(25)/25, mode='same')

    # Calculate the rate of change (first derivative) to find SUDDEN spikes
    rms_diff = np.diff(rms_smoothed, prepend=rms_smoothed[0])

    # Dynamic threshold based on the rate of change
    diff_threshold = np.mean(rms_diff) + (np.std(rms_diff) * threshold_multiplier)

    # We still want the overall volume to be above average (to avoid counting sudden quiet noises)
    min_volume_threshold = np.mean(rms_smoothed)

    # Find peaks in the rate of change (sudden hits, bat sounds)
    distance_frames = int((22050 // 512) * distance_sec)
    sudden_peaks, _ = find_peaks(rms_diff, height=diff_threshold, distance=distance_frames)

    # Filter sudden peaks to ensure the absolute volume is also sufficient
    valid_sudden_peaks = [p for p in sudden_peaks if rms_smoothed[p] > min_volume_threshold]

    # NEW: Catch dropped catches or loud appeals via sustained high energy (crowd reaction)
    # We check directly for prominent peaks in the raw volume (smoothed RMS)
    volume_threshold = np.mean(rms_smoothed) + (np.std(rms_smoothed) * 2.5)
    sustained_peaks, _ = find_peaks(rms_smoothed, height=volume_threshold, distance=distance_frames, prominence=np.std(rms_smoothed))

    # Combine sudden impact peaks and sustained high volume peaks
    all_peaks = set(valid_sudden_peaks).union(set(sustained_peaks))
    sorted_peaks = sorted(list(all_peaks))

    highlight_times = librosa.frames_to_time(sorted_peaks, sr=sr)
    spikes_list = highlight_times.tolist()

    print(f"Audio Module: Detected {len(spikes_list)} potential excitement spikes (hits/appeals).")
    return spikes_list


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



In [ ]:
pip install easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 25.2 MB/s eta 0:00:00


In [ ]:
import cv2
import easyocr
import re
import numpy as np

class ScoreboardDetector:
    def __init__(self, gpu=True):
        """
        Initializes the OCR reader.
        Note: The first time this runs, it will download the EasyOCR model to your computer.
        """
        print("OCR Module: Loading Engine...")
        self.reader = easyocr.Reader(['en'], gpu=gpu)
        self.roi = None # Will store (x, y, w, h) of the scoreboard
        print("OCR Module: Engine Ready.")

    def calibrate(self, frame):
        """
        Examines a full video frame to automatically locate the scoreboard.
        It looks for a cricket score format (e.g., '123/4' or '123-4').
        Returns True if found and calibrated successfully.
        """
        print("OCR Module: Calibrating on anchor frame...")

        # 1. Run full-frame OCR
        results = self.reader.readtext(frame)

        # 2. Regex to find "Number followed by a slash or dash, followed by a Number"
        score_pattern = re.compile(r'\d+\s*[\/-]\s*\d+')

        for bbox, text, prob in results:
            if score_pattern.search(text):
                xs = [int(pt[0]) for pt in bbox]
                ys = [int(pt[1]) for pt in bbox]

                x_min, x_max = min(xs), max(xs)
                y_min, y_max = min(ys), max(ys)

                padding_x_left = 100
                padding_x_right = 100
                padding_y_top = 30
                padding_y_bottom = 30

                h, w = frame.shape[:2]

                final_x = max(0, x_min - padding_x_left)
                final_y = max(0, y_min - padding_y_top)
                final_w = min(w, x_max + padding_x_right) - final_x
                final_h = min(h, y_max + padding_y_bottom) - final_y

                self.roi = (final_x, final_y, final_w, final_h)

                print(f"OCR Module: ✅ Calibration successful! Score found: '{text}'")
                print(f"OCR Module: 🔒 Scoreboard ROI Locked at Coordinate Box: {self.roi}")
                return True

        print("OCR Module: ❌ Calibration failed. Could not find a recognizable score format in this frame.")
        return False

    def read_score(self, frame):
        """
        Highly optimized reading phase. Cuts out the pre-found ROI and runs OCR only on that tiny slice.
        """
        if not self.roi:
            raise ValueError("Detector has not been calibrated yet! Call calibrate() on a good frame first.")

        x, y, w, h = self.roi
        cropped_roi = frame[y:y+h, x:x+w]

        # Fast OCR directly on the slice
        results = self.reader.readtext(cropped_roi)
        # Return all text extracted from the scoreboard
        detected_text = [text for bbox, text, prob in results]
        return " | ".join(detected_text).strip()

    def parse_score(self, text):
        """
        Extracts runs and wickets from text like '123/4' or '123-4'
        """
        score_pattern = re.compile(r'(\d+)\s*[\/-]\s*(\d+)')
        match = score_pattern.search(text)
        if match:
            v1 = int(match.group(1))
            v2 = int(match.group(2))

            # Common cricket format: runs/wickets or wickets/runs.
            # Usually runs > wickets, or wickets <= 10.
            if v2 > 10 and v1 <= 10:
                runs, wickets = v2, v1
            else:
                runs, wickets = v1, v2
            return runs, wickets
        return None, None

    def read_overs(self, frame):
        """
        Runs OCR on the ROI and extracts the overs formatted as 'X.Y'.
        Returns the parsed float overs or None if not found.
        """
        if not self.roi:
            return None

        x, y, w, h = self.roi
        cropped_roi = frame[y:y+h, x:x+w]

        results = self.reader.readtext(cropped_roi)

        # Regex to find overs e.g. 14.3 or 49.5
        overs_pattern = re.compile(r'\b([0-4]?\d|50)\.[0-6]\b')

        for bbox, text, prob in results:
            # Replaces commas with dots just in case OCR reads 14,3 instead of 14.3
            text_cleaned = text.replace(',', '.')
            match = overs_pattern.search(text_cleaned)
            if match:
                try:
                    return float(match.group(0))
                except ValueError:
                    pass
        return None

    def detect_keyword(self, frame, keyword):
        """
        Detects if a specific keyword (case-insensitive) is present in the scoreboard ROI.
        """
        if not self.roi:
            # Try to calibrate if not already calibrated. If it still fails, return False.
            if not self.calibrate(frame):
                return False

        x, y, w, h = self.roi
        cropped_roi = frame[y:y+h, x:x+w]
        results = self.reader.readtext(cropped_roi)
        for bbox, text, prob in results:
            if keyword.lower() in text.lower():
                return True
        return False

def validate_highlight(frame_before, frame_after, detector):
    """
    Returns True if the highlight is confirmed (i.e., we detected a score difference of 4, 6 or a wicket fell).
    """
    try:
        if not detector.roi:
            success = detector.calibrate(frame_before)
            if not success:
                print("OCR Validation: Could not calibrate scoreboard ROI. Assuming false positive.")
                return False

        text_before = detector.read_score(frame_before)
        text_after = detector.read_score(frame_after)

        # We need something to compare against
        if text_before == "":
            print("OCR Validation: Couldn't read score on 'before' frame.")
            return False

        if text_before != text_after and len(text_after) > 0:
            runs_b, wick_b = detector.parse_score(text_before)
            runs_a, wick_a = detector.parse_score(text_after)

            if runs_b is not None and runs_a is not None:
                r_diff = runs_a - runs_b
                w_diff = wick_a - wick_b
                if r_diff in [4, 6] or w_diff > 0:
                    print(f"OCR Validation: Match! Great play detected: Runs diff = {r_diff}, Wickets diff = {w_diff}")
                    return True
                else:
                    print(f"OCR Validation: Failed. Score changed but not by 4, 6, or a wicket.")
                    return False

            print(f"OCR Validation: Match by fallback! Score changed from '{text_before}' to '{text_after}'")
            return True
        else:
            print(f"OCR Validation: Failed. Score unchanged ('{text_before}' vs '{text_after}')")
            return False

    except Exception as e:
        print(f"OCR Validation failed with error: {e}")
        return False

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import os

def detect_deliveries(video_path, model, fps_to_sample=2, threshold=0.5):
    """
    Detects deliveries in a video based on bowler run-up.

    Args:
        video_path (str): Path to the video file.
        model: The pre-loaded Keras model for run-up detection.
        fps_to_sample (int): How many frames per second to actually pass to the CNN to save compute.
        threshold (float): Confidence threshold for the binary classifier.

    Returns:
        list: A list of timestamps (in seconds) indicating the start of a recognized delivery run-up.
    """
    print(f"Vision Module: Processing video {video_path}...")
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    video_fps = cap.get(cv2.CAP_PROP_FPS)
    if video_fps == 0 or np.isnan(video_fps):
        video_fps = 30.0 # fallback approximation

    # How many actual video frames to skip between predictions
    frame_interval = int(video_fps / fps_to_sample) if video_fps > fps_to_sample else 1

    delivery_times = []
    in_runup = False

    frame_count = 0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Vision Module: Total frames in video: {total_frames} @ {video_fps} FPS. Sampling every {frame_interval} frames.")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % frame_interval == 0:
            current_sec = frame_count / video_fps

            # Preprocess the frame
            # cv2 read as BGR, model trained on MobileNetV2 uses RGB, normalized to [0,1]
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (224, 224))
            img_array = np.array(img, dtype=np.float32) / 255.0
            img_batch = np.expand_dims(img_array, axis=0) # Shape: (1, 224, 224, 3)

            # Predict
            pred = model.predict(img_batch, verbose=0)[0][0]

            if pred >= threshold:
                if not in_runup:
                    in_runup = True
                    delivery_times.append(current_sec)
                    print(f"Vision Module: Run-up detected at {current_sec:.1f}s")
            else:
                in_runup = False

        frame_count += 1

    cap.release()
    print(f"Vision Module: Detected {len(delivery_times)} deliveries in total.")

    return delivery_times

def detect_runup_in_window(video_path, model, start_time=0.0, end_time=10.0, fps_to_sample=2, threshold=0.5):
    """
    Scans a specific window of a video to find the exact start time of a bowler run-up.
    Returns the timestamp of the first frame that exceeds the threshold, or None if no run-up is found.
    """
    print(f"Vision Module: Starting targeted CNN scan from {start_time:.1f}s to {end_time:.1f}s...")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    video_fps = cap.get(cv2.CAP_PROP_FPS)
    if video_fps == 0 or np.isnan(video_fps):
        video_fps = 30.0

    frame_interval = int(video_fps / fps_to_sample) if video_fps > fps_to_sample else 1

    start_frame = int(max(0, start_time) * video_fps)
    end_frame = int(end_time * video_fps)

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    current_frame_count = start_frame

    found_time = None

    while current_frame_count <= end_frame:
        ret, frame = cap.read()
        if not ret:
            break

        if current_frame_count % frame_interval == 0:
            current_sec = current_frame_count / video_fps

            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (224, 224))
            img_array = np.array(img, dtype=np.float32) / 255.0
            img_batch = np.expand_dims(img_array, axis=0)

            pred = model.predict(img_batch, verbose=0)[0][0]

            if pred >= threshold:
                found_time = current_sec
                print(f"Vision Module: Targeted run-up pinpointed at {current_sec:.1f}s!")
                break

        current_frame_count += 1

    cap.release()
    return found_time

### Recompiling the Keras Model

It's a good practice to recompile and re-save Keras models if you suspect compatibility issues with different TensorFlow versions. This ensures the model's metadata and internal structure are up-to-date with the current environment. Here, we'll load the `cricket_runup_final.h5` model, recompile it with a placeholder optimizer and loss (as these don't affect inference but are needed for saving a 'compiled' model), and then save it back, overwriting the old one.

In [ ]:
import tensorflow as tf
import os

MODEL_PATH = '/content/drive/MyDrive/cricket_runup_final.h5'

if os.path.exists(MODEL_PATH):
    print(f"Defining model architecture and loading weights from {MODEL_PATH}...")

    # Define the model architecture explicitly
    # Assuming a MobileNetV2 base with a binary classification head,
    # based on common practices for such tasks and the input shape (224,224,3).
    base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3),
                                                   include_top=False,
                                                   weights=None) # We will load weights from our .h5 file

    x = base_model.output
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    output = tf.keras.layers.Dense(1, activation='sigmoid')(x) # Binary classification

    model = tf.keras.models.Model(inputs=base_model.input, outputs=output)

    # Load only the weights from the H5 file into the new model architecture
    # `by_name=True` helps if layer names in the new architecture don't exactly match the old,
    # `skip_mismatch=True` ignores layers in the H5 that don't match the new model's architecture.
    try:
        model.load_weights(MODEL_PATH, by_name=True, skip_mismatch=True)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"Error loading weights: {e}")
        print("Attempting to load weights directly, this might still fail if layers are too different.")
        # Fallback to direct load_weights if by_name/skip_mismatch causes issues (less common)
        model.load_weights(MODEL_PATH)

    # Recompile the model. For a binary classification model, a common setup is:
    # optimizer='adam', loss='binary_crossentropy', metrics=['accuracy']
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    print("Model recompiled successfully with loaded weights.")
else:
    print(f"Model file not found at {MODEL_PATH}. Cannot proceed with loading weights.")

Defining model architecture and loading weights from /content/drive/MyDrive/cricket_runup_final.h5...
Weights loaded successfully.
Model recompiled successfully with loaded weights.


In [ ]:
import os
import cv2
import numpy as np
from moviepy.editor import VideoFileClip, concatenate_videoclips

def merge_intervals(intervals, buffer_sec=2):
    """
    Merges overlapping time intervals, considering a small buffer.
    intervals: list of (start, end) tuples
    buffer_sec: additional seconds to consider for merging (e.g., if intervals are close)
    """
    if not intervals:
        return []

    sorted_intervals = sorted(intervals, key=lambda x: x[0])
    merged = []
    current_start, current_end = sorted_intervals[0]

    for next_start, next_end in sorted_intervals[1:]:
        if next_start <= current_end + buffer_sec:
            current_end = max(current_end, next_end)
        else:
            merged.append((current_start, current_end))
            current_start, current_end = next_start, next_end

    merged.append((current_start, current_end))
    return merged

def main():
    VIDEO_INPUT = '/content/drive/MyDrive/Cricket_match.mp4'
    MODEL_PATH = '/content/drive/MyDrive/cricket_runup_final.h5'
    OUTPUT_FILE = '/content/drive/MyDrive/final_cricket_highlights01.mp4'

    print("="*50)
    print("🏏 CRICKET HIGHLIGHT GENERATOR (OCR-FIRST) 🏏")
    print("="*50)

    if not os.path.exists(VIDEO_INPUT):
        print(f"Warning: Please ensure '{VIDEO_INPUT}' is in the current directory!")
        return

    # The model variable is now global due to cell 5b19166e
    # We need to make sure cell 5b19166e has been run to load `model`
    if 'model' not in globals():
        print("Error: Keras model not loaded. Please run the model loading/recompilation cell (cell 5b19166e) first.")
        return

    # STEP 1: Process Audio (The Excitement Filter)
    print("\n--- STEP 1: AUDIO ANALYSIS ---")
    spike_times = get_audio_spikes(VIDEO_INPUT, distance_sec=10, threshold_multiplier=3.0)
    if not spike_times:
        print("No audio spikes detected. Exiting.")
        return

    # STEP 2 & 3: OCR Validation Around Audio Spikes
    print("\n--- STEP 2: OCR VALIDATION AROUND AUDIO SPIKES ---")
    detector = ScoreboardDetector(gpu=True)

    cap = cv2.VideoCapture(VIDEO_INPUT)
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    if video_fps == 0 or np.isnan(video_fps):
        video_fps = 30.0

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    video_duration = total_frames / video_fps

    # We calibrate on the first valid frame
    ret, initial_frame = cap.read()
    if not ret or not detector.calibrate(initial_frame):
        print("Warning: Could not calibrate OCR on the very first frame. Will try to re-calibrate automatically.")

    confirmed_highlights = []

    for s_time in spike_times:
        print(f"\nChecking audio spike at {s_time:.1f}s...")

        # Check frame 10 seconds before spike and 15 seconds after
        time_before = max(0, s_time - 10)
        time_after = min(video_duration, s_time + 15)

        cap.set(cv2.CAP_PROP_POS_FRAMES, int(time_before * video_fps))
        ret_b, frame_b = cap.read()
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(time_after * video_fps))
        ret_a, frame_a = cap.read()

        if not ret_b or not ret_a:
            continue

        if not detector.roi:
            detector.calibrate(frame_b)

        is_highlight = False

        # 1. Check for overs
        over_b = detector.read_overs(frame_b)
        over_a = detector.read_overs(frame_a)

        if over_b is not None and over_a is not None and over_a > over_b:
            print(f"OCR Validation: Over incremented `{over_b}` -> `{over_a}`.")
            is_highlight = True

        # 2. Check for score changing (4, 6, wicket)
        if not is_highlight:
            try:
                score_b = detector.read_score(frame_b)
                r_b, w_b = detector.parse_score(score_b)

                score_a = detector.read_score(frame_a)
                r_a, w_a = detector.parse_score(score_a)

                if r_b is not None and r_a is not None:
                    r_diff = r_a - r_b
                    w_diff = w_a - w_b
                    if r_diff in [4, 6] or w_diff > 0:
                        print(f"OCR Validation: Highlight Event! Runs +{r_diff}, Wickets +{w_diff}.")
                        is_highlight = True
            except Exception:
                pass

        # 3. Check for 'WIN' text (Match End)
        if not is_highlight:
            if detector.detect_keyword(frame_a, keyword='WIN'):
                print("OCR Validation: Detected 'WIN' text. Match end highlight.")
                is_highlight = True

        # 4. Check for over value 0.0 (Innings End/Start)
        if not is_highlight:
            if over_a == 0.0 and over_b is not None and over_b != 0.0:
                print("OCR Validation: Detected over value reset to 0.0. Innings end/start highlight.")
                is_highlight = True

        if is_highlight:
            print("=> Highlight Confirmed!")
            # Use time_after as the anchor where graphics are updated (equivalent to b_time previously)
            confirmed_highlights.append(time_after)
        else:
            print("=> False Alarm. No significant scoreboard change.")

    cap.release()
    print(f"\nFiltered down to {len(confirmed_highlights)} exciting, confirmed deliveries.")

    if not confirmed_highlights:
        print("No confirmed exciting deliveries found. Exiting.")
        return

    # STEP 4: Pinpointing Run-up with CNN
    print("\n--- STEP 4: PINPOINTING RUN-UP (TARGETED CNN) ---")
    perfect_clips_starts = []

    for idx, b_time in enumerate(confirmed_highlights):
        print(f"\nTargeting CNN for Highlight {idx+1} (Ball ending near {b_time:.1f}s)...")
        # Increased buffer for scan_start to check earlier
        scan_start = max(0, b_time - 35)
        scan_end = b_time

        runup_start = detect_runup_in_window(
            video_path=VIDEO_INPUT,
            model=model,
            start_time=scan_start,
            end_time=scan_end,
            fps_to_sample=2,
            threshold=0.5
        )

        if runup_start is not None:
            perfect_clips_starts.append((runup_start, b_time))
        else:
            print(f"Warning: CNN couldn't find run-up in the expected window. Utilizing math fallback.")
            perfect_clips_starts.append((scan_start + 5, b_time)) # Rough estimate backup

    # STEP 5: Intelligent Clipping & Merging
    print("\n--- STEP 5: CLIPPING & STITCHING ---")
    video = VideoFileClip(VIDEO_INPUT)

    # Create effective clip boundaries to merge, allowing for lead-in/lead-out
    raw_clips_for_merging = []
    for clip_start_event, clip_end_event in perfect_clips_starts:
        t_start_effective = max(0, clip_start_event - 2) # Lead in 2s before the exact true run-up
        t_end_effective = min(video.duration, clip_end_event + 3) # Show play settling and graphic updating
        raw_clips_for_merging.append((t_start_effective, t_end_effective))

    print(f"\n--- Consolidating {len(raw_clips_for_merging)} raw highlight segments ---")
    final_merged_clips = merge_intervals(raw_clips_for_merging, buffer_sec=5) # Allow 5s buffer for merging
    print(f"Consolidated into {len(final_merged_clips)} final highlight segments.")

    clips = []
    for i, (clip_start_final, clip_end_final) in enumerate(final_merged_clips):
        print(f"Creating Final Clip {i+1}: {clip_start_final:.1f}s to {clip_end_final:.1f}s")
        clip = video.subclip(clip_start_final, clip_end_final)
        clips.append(clip)

    if clips:
        print("\nStitching final video together...")
        final_highlights = concatenate_videoclips(clips)
        final_highlights.write_videofile(OUTPUT_FILE, codec="libx264", audio_codec="aac")
        print(f"\n✅ SUCCESS: '{OUTPUT_FILE}' has been generated using OCR-First Pipeline!")

if __name__ == "__main__":
    main()

🏏 CRICKET HIGHLIGHT GENERATOR (OCR-FIRST) 🏏

--- STEP 1: AUDIO ANALYSIS ---
Audio Module: Extracting audio stream from video...
Audio Module: Analyzing audio energy for sudden spikes...
Audio Module: Detected 366 potential excitement spikes (hits/appeals).

--- STEP 2: OCR VALIDATION AROUND AUDIO SPIKES ---
OCR Module: Loading Engine...
OCR Module: Engine Ready.
OCR Module: Calibrating on anchor frame...
OCR Module: ❌ Calibration failed. Could not find a recognizable score format in this frame.

Checking audio spike at 57.7s...
OCR Module: Calibrating on anchor frame...
OCR Module: ❌ Calibration failed. Could not find a recognizable score format in this frame.
OCR Module: Calibrating on anchor frame...
OCR Module: ❌ Calibration failed. Could not find a recognizable score format in this frame.
=> False Alarm. No significant scoreboard change.

Checking audio spike at 70.7s...
OCR Module: Calibrating on anchor frame...
OCR Module: ❌ Calibration failed. Could not find a recognizable score

MoviePy - Done.
Moviepy - Writing video /content/drive/MyDrive/final_cricket_highlights01.mp4



Moviepy - Done !
Moviepy - video ready /content/drive/MyDrive/final_cricket_highlights01.mp4

✅ SUCCESS: '/content/drive/MyDrive/final_cricket_highlights01.mp4' has been generated using OCR-First Pipeline!
